In [50]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import math

spark = SparkSession.builder.appName("Lab 1").getOrCreate()

In [51]:
trips = spark.read.csv("trips.csv", header=True, inferSchema=True)
stations = spark.read.csv("stations.csv", header=True, inferSchema=True)

trips.show(5)
# trips.printSchema()

stations.show(5)
# stations.printSchema()

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|           NULL|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|    NULL|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

In [52]:
bike_total_time = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration")) \
    .orderBy(F.desc("total_duration"))

max_bike = bike_total_time.first()
print(f"Bike ID: {max_bike['bike_id']}, Total Duration: {max_bike['total_duration']} sec")

Bike ID: 535, Total Duration: 18611693 sec


In [53]:
!pip install geopy

In [54]:
from geopy.distance import geodesic
from pyspark.sql import functions as F

def calculate_distance(lat1, lon1, lat2, lon2):
    try:
        return float(geodesic((lat1, lon1), (lat2, lon2)).kilometers)
    except:
        return None

distance_udf = F.udf(calculate_distance, FloatType())

stations_a = stations.alias("a").select(
    F.col("id").alias("a_id"),
    F.col("lat").alias("lat_a"),
    F.col("long").alias("lon_a")
)

stations_b = stations.alias("b").select(
    F.col("id").alias("b_id"),
    F.col("lat").alias("lat_b"),
    F.col("long").alias("lon_b")
)

distance_df = (
    stations_a.crossJoin(stations_b.hint("broadcast"))
    .filter("a_id < b_id")  # Исключаем дубликаты
    .withColumn("distance", distance_udf("lat_a", "lon_a", "lat_b", "lon_b"))
    .orderBy(F.desc("distance"))
)

max_distance_row = distance_df.first()
print(f"Max Distance: {max_distance_row['distance']:.2f} km between stations {max_distance_row['a_id']} and {max_distance_row['b_id']}")

Max Distance: 69.92 km between stations 16 and 60


In [55]:
from pyspark.sql.functions import unix_timestamp

trips_with_cast_time = trips_df_spark.withColumn(
    "start_time",
    F.unix_timestamp(F.col("start_date"), "M/d/yyyy H:mm")
).withColumn(
    "end_time",
    F.unix_timestamp(F.col("end_date"), "M/d/yyyy H:mm")
).withColumn(
    "duration_minutes",
    (F.col("end_time") - F.col("start_time")) / 60
)

max_bike_id = max_bike["bike_id"]
max_bike_trips = trips_with_cast_time.filter(col("bike_id") == max_bike_id).orderBy("start_time")

route = []
for row in max_bike_trips.select("start_station_name", "end_station_name").collect():
    route.append(row["start_station_name"])
    route.append(row["end_station_name"])

route = list(dict.fromkeys(route))

print(f"Route of bike: {max_bike_id}")
# print(" -> ".join(route))
print("[First 5 stations]")
print(" -> ".join(route[:5]))
print("[Last 5 stations]")
print(" -> ".join(route[-5:]))

Route of bike: 535
[First 5 stations]
Post at Kearney -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome -> 2nd at South Park
[Last 5 stations]
Embarcadero at Folsom -> South Van Ness at Market -> Broadway St at Battery St -> Post at Kearny -> Washington at Kearny


In [56]:
unique_bikes = trips.select("bike_id").distinct().count()
print(f"Unique bikes: {unique_bikes}")

Unique bikes: 700


In [57]:
user_duration = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration")) \
    .filter(F.col("total_duration") > 3 * 3600) \
    .orderBy(F.desc("total_duration"))

user_duration.show()

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
|    466|       3933272|
|    613|       2409014|
|    526|       2253019|
|    415|       2248886|
|    572|       2234149|
|    524|       2214314|
|    542|       2213422|
|    465|       2185170|
|    376|       2178177|
|    371|       2147355|
|    484|       2126619|
|    593|       2121705|
|    630|       2103546|
|    591|       2077782|
|    634|       2075880|
|    382|       2074399|
|    503|       2072630|
|    518|       2067178|
|    375|       2041246|
+-------+--------------+
only showing top 20 rows

